# Support Vector Classifiers

Here we try many support vector classifiers, with different kernels. 

In [1]:
import pandas as pd

learn_data = pd.read_csv("resampled_train.csv", header = None)
learn_data.columns = ['Age', 'TB', 'DB', 'Alkphos', 'Sgpt', 'Sgot', 'TP', 'ALB', 'AR', 'BilRatio', 'Female', 'Target']
learn_data["Female"] = learn_data["Female"].astype("category")
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,AR,BilRatio,Female,Target
0,0.173844,1.109406,1.197304,0.362037,-1.391442,0.487504,0.529868,-0.903219,-1.399329,1.203258,0,0
1,-0.379308,0.226878,0.469206,-0.573270,0.069730,0.288250,0.908406,1.482342,1.488320,0.927290,0,0
2,-1.362690,-0.430090,-0.383317,-0.232378,0.039705,0.575302,-0.227207,-0.024328,0.212382,-0.353374,0,0
3,-0.194924,-0.795164,-0.697958,-0.925510,-0.157437,0.589292,-0.227207,0.101228,0.413846,-0.458710,1,0
4,0.542612,2.761282,2.439451,1.783803,-0.349504,-0.293097,1.286944,0.352339,-0.459164,1.153956,1,0


In [2]:
from sklearn.model_selection import train_test_split

X = learn_data.drop(columns = ["Target"])
Xnum = X.drop(columns = ["Female"])
y = learn_data["Target"]

X_train, X_val, Xnum_train, Xnum_val, y_train, y_val = train_test_split(X, Xnum, y, test_size = 0.33, random_state = 42)

## Metrics

In [3]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

metrics_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

## Linear kernel (or no kernel)

We have seen with other linear classifiers that the performance is not very good because of two reasons:
- Excessive resampling: we might be resampling too much, and this may affect our predictive power by creating samples that do not correspond to the real data.

In [4]:
from sklearn.svm import LinearSVC

linear_model = LinearSVC()
linear_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(linear_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	189	36
	0	79	128
Accuracy: 73.38%


In [5]:
confusion(np.array(y_val), pd.Series(linear_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	81	17
	0	49	67
Accuracy: 69.16%


In [6]:
from sklearn.model_selection import cross_validate

cross_val_results = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

Cs = np.logspace(start = -1, stop = 2, num = 50)

for C in Cs:
    linear_model = LinearSVC(C = C)
    this_results = pd.DataFrame(cross_validate(linear_model, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
    mean_results = this_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
    cross_val_results.loc[C, :] = mean_results

cross_val_results.sort_values(by = "F1 Macro", ascending = False).head()

,F1 Macro,Recall,Precision,Accuracy
0.954095,0.720552,0.724639,0.73844,0.72452
0.175751,0.720399,0.724591,0.738882,0.72452
0.202359,0.718964,0.723053,0.736635,0.72297
2.222996,0.718917,0.723077,0.73692,0.72297
1.676833,0.718917,0.723077,0.73692,0.72297


In [7]:
linear_model_best = LinearSVC(C = 1)
cross_val_results = pd.DataFrame(cross_validate(linear_model_best, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["Linear-best", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Linear-best,0.720552,0.724639,0.73844,0.72452


## Gaussian kernel

In [8]:
from sklearn.svm import SVC

rbf_scale_model = SVC(kernel = "rbf", gamma = "scale")
rbf_scale_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(rbf_scale_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	214	11
	0	77	130
Accuracy: 79.63%


In [9]:
confusion(np.array(y_val), pd.Series(rbf_scale_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	87	11
	0	50	66
Accuracy: 71.50%


In [10]:
rbf_scale_model = SVC(kernel = "rbf", gamma = "scale")
cross_val_results = pd.DataFrame(cross_validate(rbf_scale_model, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["Gaussian-Scale", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Gaussian-Scale,0.736766,0.743077,0.76988,0.743113
Linear-best,0.720552,0.724639,0.73844,0.72452


We can also use the automatic $\gamma$, which is $\gamma = \frac{1}{n}$. When doing simple train-validation, this gives us slightly worse results, but it isn't very 

In [11]:
rbf_auto_model = SVC(kernel = "rbf", gamma = "auto")
rbf_auto_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(rbf_auto_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	214	11
	0	78	129
Accuracy: 79.40%


In [12]:
confusion(np.array(y_val), pd.Series(rbf_auto_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	87	11
	0	51	65
Accuracy: 71.03%


In [13]:
rbf_auto_model = SVC(kernel = "rbf", gamma = "auto")
cross_val_results = pd.DataFrame(cross_validate(rbf_auto_model, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["Gaussian-Auto", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Gaussian-Scale,0.736766,0.743077,0.76988,0.743113
Gaussian-Auto,0.731753,0.738438,0.765874,0.738462
Linear-best,0.720552,0.724639,0.73844,0.72452


We can also try to find the best value of $C$, using the scaled value of $\gamma$.

In [14]:
import warnings
warnings.filterwarnings("ignore")

cross_val_results = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

Cs = np.logspace(start = -1, stop = 2, num = 50)

for C in Cs:
    rbf_model = SVC(kernel = "rbf", C = C, gamma = "scale")
    this_results = pd.DataFrame(cross_validate(rbf_model, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
    mean_results = this_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
    cross_val_results.loc[C, :] = mean_results

cross_val_results.sort_values(by = "F1 Macro", ascending = False).head()

,F1 Macro,Recall,Precision,Accuracy
86.851137,0.813375,0.815817,0.837047,0.815874
100.000000,0.81176,0.814279,0.835785,0.814323
65.512856,0.808595,0.811154,0.832345,0.811234
75.431201,0.808541,0.81113,0.832609,0.811222
56.898660,0.803618,0.806514,0.828855,0.806583


In [15]:
rbf_model_best = SVC(kernel = "rbf", C = 90, gamma = "scale")
cross_val_results = pd.DataFrame(cross_validate(rbf_model_best, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["Gaussian-Scale-Best", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Gaussian-Scale-Best,0.813375,0.815817,0.837047,0.815874
Gaussian-Scale,0.736766,0.743077,0.76988,0.743113
Gaussian-Auto,0.731753,0.738438,0.765874,0.738462
Linear-best,0.720552,0.724639,0.73844,0.72452


## Polynomial kernel

In [16]:
poly_model = SVC(kernel = "poly", degree = 2, gamma = "scale")
poly_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(poly_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	213	12
	0	102	105
Accuracy: 73.61%


In [17]:
confusion(np.array(y_val), pd.Series(poly_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	80	18
	0	67	49
Accuracy: 60.28%


In [18]:
cross_val_results = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

Cs = np.logspace(start = -1, stop = 2, num = 50)
degrees = [2, 3, 4, 5]

for degree in degrees:
    for C in Cs:
        poly_model = SVC(kernel = "poly", C = C, degree = degree, gamma = "scale")
        this_results = pd.DataFrame(cross_validate(poly_model, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
        mean_results = this_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
        cross_val_results.loc[f"Degree: {degree} - C: {C} ", :] = mean_results

cross_val_results.sort_values(by = "F1 Macro", ascending = False).head()

,F1 Macro,Recall,Precision,Accuracy
Degree: 4 - C: 100.0,0.747268,0.752332,0.779303,0.752451
Degree: 4 - C: 86.8511373751352,0.745388,0.750793,0.77998,0.750924
Degree: 4 - C: 75.43120063354614,0.737636,0.743053,0.770537,0.743196
Degree: 4 - C: 65.51285568595509,0.734592,0.739976,0.76705,0.740107
Degree: 4 - C: 32.374575428176435,0.730619,0.73851,0.775297,0.738557


In [19]:
poly_model_best = SVC(kernel = "poly", C = 100, degree = 4, gamma = "scale")
cross_val_results = pd.DataFrame(cross_validate(poly_model_best, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["Poly-Degree4", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Gaussian-Scale-Best,0.813375,0.815817,0.837047,0.815874
Poly-Degree4,0.747268,0.752332,0.779303,0.752451
Gaussian-Scale,0.736766,0.743077,0.76988,0.743113
Gaussian-Auto,0.731753,0.738438,0.765874,0.738462
Linear-best,0.720552,0.724639,0.73844,0.72452


## Sigmoid

In [20]:
sig_model = SVC(kernel = "sigmoid", gamma = "scale")
sig_model.fit(Xnum_train, y_train)

confusion(np.array(y_train), pd.Series(sig_model.predict(Xnum_train)))

		Predicted
		+1	0
Real	+1	169	56
	0	88	119
Accuracy: 66.67%


In [21]:
confusion(np.array(y_val), pd.Series(sig_model.predict(Xnum_val)))

		Predicted
		+1	0
Real	+1	79	19
	0	44	72
Accuracy: 70.56%


In [22]:
cross_val_results = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

Cs = np.logspace(start = -1, stop = 2, num = 50)

for C in Cs:
    sig_model = SVC(kernel = "sigmoid", C = C, gamma = "scale")
    this_results = pd.DataFrame(cross_validate(sig_model, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
    mean_results = this_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
    cross_val_results.loc[C, :] = mean_results

cross_val_results.sort_values(by = "F1 Macro", ascending = False).head()

,F1 Macro,Recall,Precision,Accuracy
0.175751,0.712772,0.719832,0.743918,0.719833
0.232995,0.712137,0.718341,0.739461,0.718318
0.202359,0.711847,0.718341,0.740178,0.718306
0.100000,0.708434,0.716779,0.74507,0.716744
0.268270,0.707817,0.713702,0.733147,0.713679


In [23]:
cross_val_results = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

coeff0s = np.logspace(start = -1, stop = 1, num = 50)

for i in [-1, +1]:
    for coeff0 in coeff0s:
        sig_model = SVC(kernel = "sigmoid", C = 1, coef0 = i * coeff0, gamma = "scale")
        this_results = pd.DataFrame(cross_validate(sig_model, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
        mean_results = this_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
        cross_val_results.loc[i * coeff0, :] = mean_results

cross_val_results.sort_values(by = "F1 Macro", ascending = False).head()

,F1 Macro,Recall,Precision,Accuracy
-1.526418,0.71159,0.722933,0.764615,0.722946
-1.676833,0.710903,0.722909,0.768114,0.722946
-0.719686,0.709494,0.719784,0.757317,0.719869
-1.389495,0.708721,0.719832,0.759438,0.719845
-1.264855,0.708721,0.719832,0.759438,0.719845


## Trying our best models on test dataset


In [24]:
test_data = pd.read_csv("scaled_test.csv", header = None)
test_data.columns = ['Age', 'TB', 'DB', 'Alkphos', 'Sgpt', 'Sgot', 'TP', 'ALB', 'AR', 'BilRatio', 'Female']
test_data["Female"] = learn_data["Female"].astype("category")
test_data.head()

,Age,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,AR,BilRatio,Female
0,-2.100226,-0.795164,-1.235840,1.907027,-0.527803,-0.567456,0.624503,1.356786,1.555475,-1.512071,0
1,1.034303,0.171538,0.469206,-0.117670,0.688275,1.320148,2.044019,1.105674,-0.459164,1.121330,0
2,0.911380,-0.795164,-0.697958,-0.643898,-0.269091,-1.387576,1.286944,1.356786,0.548155,-0.458710,0
3,0.911380,1.351361,1.349951,-0.212816,2.914717,3.236675,0.813772,0.101228,-0.526319,1.056650,1
4,0.173844,-0.537931,-0.697958,-0.631959,-0.627534,0.132669,-0.889648,-0.526551,-0.123391,-0.926871,1


In [27]:
test_data_num = test_data.drop(columns = ["Female"])

### Gaussian kernel (no categorical)

In [28]:
rbf_model_best = SVC(kernel = "rbf", C = 90, gamma = "scale")
rbf_model_best.fit(Xnum, y)

labels_rbf = pd.DataFrame(columns = ['ID', 'Label'])
labels_rbf['Label'] = pd.DataFrame(rbf_model_best.predict(test_data_num))
labels_rbf['ID'] = labels_rbf.index + 1
labels_rbf

,ID,Label
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0
...,...,...
111,112,0
112,113,0
113,114,0
114,115,0


In [29]:
labels_rbf.Label.value_counts()

Label
0    87
1    29
Name: count, dtype: int64

In [30]:
labels_rbf.to_csv('new_predictions/svm_rbf_best.csv', index = False)